# Experimento A2: entorno determinista con tiempo en el estado

### Carga de librerías

In [1]:
import numpy as np
import pandas as pd
import random

import entorno
import agentes
import train

### Parámetros del entorno

In [2]:
TAMANOS = {
    'n_estaciones': 2,
    'anclas_por_est': 10,
    'bicis_por_est': 5,    # Estado inicial
    'bicis_por_accion': 2  # Bicis movidas por el agente en c/ accion
}
COSTOS = {
    'accion': 1,
    'accion_invalida': 10,
    'est_vacia': 1,
    'est_llena': 1,
}
TIEMPOS = {
    'viaje': 0,
    'accion': 0,
    'episodio': 100,
    'estado': 100,    #  T
    'update': 1       #  t_step
}
PROBS = {
    'origen': [0, 1],
    'destino': np.array(2*[[1, 0]])
}

TRAIN_PARAMS = {
    'alpha_init': 0.1,
    'alpha_step': 1.0,
    'alpha_end':  0.1,

    'eps_init': 0.1,
    'eps_step': 1.0,
    'eps_end':  0.1
}

### Entrenar agentes

In [3]:
random.seed(10)

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

agente_M = agentes.MonteCarloAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
_, _, _, _ = train.train(env, agente_M, params=TRAIN_PARAMS, N=10**5, verbose=False)

agente_Q = agentes.QLearningAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
_, _, _, _ = train.train(env, agente_Q, params=TRAIN_PARAMS, N=10**5, verbose=False)

### Visualizar resultados

In [4]:
print("MONTE CARLO")
_, _, _, _ = train.test(env, agente_M.q_table, N=1)
print()
print("Q LEARNING")
_, _, _, _ = train.test(env, agente_Q.q_table, N=1)

MONTE CARLO
Entrenamiento finalizado en 0.0 minutos.
# Insatisfechos: 1.0 ± 0.0 (1.0 ± 0.0%)
# Prolongados: 0.0 ± 0.0 (0.0 ± 0.0%)
Tiempo desbalanceo: 8.0 ± 0.0 (4.0 ± 0.0%)
Recompensa: -48.0 ± 0.0

Q LEARNING
Entrenamiento finalizado en 0.0 minutos.
# Insatisfechos: 1.0 ± 0.0 (1.0 ± 0.0%)
# Prolongados: 0.0 ± 0.0 (0.0 ± 0.0%)
Tiempo desbalanceo: 98.0 ± 0.0 (49.0 ± 0.0%)
Recompensa: -48.0 ± 0.0


### Comparar tablas de valores ($v_{\pi}$) con el óptimo teórico

In [ ]:
v_M = {(int(k[1]), int(k[2])): np.max(v) for k,v in agente_M.q_table.items()}
v_Q = {(int(k[1]), int(k[2])): np.max(v) for k,v in agente_Q.q_table.items()}

v_star = {(b, t): min(-np.ceil((100-t-b)/2), 0) for b in range(10) for t in range(100)}

estados_imposibles = [
    (0,0), (1,0), (2,0), (3,0), (4,0),        (6,0), (7,0), (8,0), (9,0),
    (0,1), (1,1),        (3,1),        (5,1),        (7,1), (8,1), (9,1),
                  (2,2),        (4,2),        (6,2),        (8,2), (9,2),
                         (3,3),        (5,3),        (7,3),        (9,3),
                                (4,4),        (6,4),        (8,4),
                                       (5,5),        (7,5),        (9,5),
                                              (6,6),        (8,6),
                                                     (7,7),        (9,7),
                                                            (8,8),
                                                                   (9,9)
]

v_star = {k: v for k,v in v_star.items() if k not in estados_imposibles}

In [42]:
s_M, s_Q, s_star = len(v_M.keys()), len(v_Q.keys()), len(v_star.keys())

print(f"Estados visitados por Monte Carlo: {s_M} de {s_star} ({100*s_M/s_star:.1f}%)")
print(f"Estados visitados por Q-Learning: {s_Q} de {s_star} ({100*s_Q/s_star:.1f}%)")

Estados visitados por Monte Carlo: 918 de 963 (95.3%)
Estados visitados por Q-Learning: 956 de 963 (99.3%)


In [46]:
sse_M, sse_Q = 0, 0
for k in v_star.keys():
    if k not in v_M.keys():
        sse_M += (v_star[k])**2
    else:
        sse_M += (v_M[k]-v_star[k])**2

    if k not in v_Q.keys():
        sse_Q += (v_star[k])**2
    else:
        sse_Q += (v_Q[k]-v_star[k])**2

rmse_M = (sse_M/963)**0.5
rmse_Q = (sse_Q/963)**0.5

print(f"RMSE Monte Carlo: {rmse_M:.1f}")
print(f"RMSE Q-Learning: {rmse_Q:.4f}")

RMSE Monte Carlo: 15.2
RMSE Q-Learning: 0.0822


### Entrenar y testear varias veces (robustez ante exploración aleatoria)

In [43]:
random.seed(123)

N = 10**5
n_sims = 10
conteo_M, conteo_Q = 0, 0

for _ in range(n_sims):
    env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)
    agente_M2 = agentes.MonteCarloAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
    agente_Q2 = agentes.QLearningAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])

    _, _, _, _ = train.train(env, agente_M2, params=TRAIN_PARAMS, N=N, verbose=False)
    _, m_M, _, _ = train.test(env, agente_M2.q_table, N=1, verbose=False)

    _, _, _, _ = train.train(env, agente_Q2, params=TRAIN_PARAMS, N=N, verbose=False)
    _, m_Q, _, _ = train.test(env, agente_Q2.q_table, N=1, verbose=False)

    if m_M['rewards'][0] == -48:
        conteo_M += 1
    if m_Q['rewards'][0] == -48:
        conteo_Q += 1

print(f"Simulaciones optimas (Monte Carlo): {100*(conteo_M/n_sims):.1f}%")
print(f"Simulaciones optimas (Q-Learning): {100*(conteo_Q/n_sims):.1f}%")

Simulaciones optimas (Monte Carlo): 40.0%
Simulaciones optimas (Q-Learning): 100.0%
